# Session 4 · Types and the Payoff

**Goal:** give every remaining column its real type (integer, exact money, time of day, yes/no, ordered codes), then rerun the Trust Test and ask questions that weren't possible before.

**How to use this notebook**
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Nothing here can break anything. If things get strange, restart (Colab: **Runtime → Restart session**; Databricks: **Run → Clear state**) and run from the top.

> ⚠️ **Mock data only.** Everything in this course is fictional. Never load real offender data into Colab or any tool that isn't approved for criminal justice information.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and builds today's data.
SESSION = 4
REPO_URL = "https://github.com/YOUR-ORG/lunch-and-learn-data-cleansing"

import os, subprocess, sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # Databricks Git folder or local copy
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    colab_copy = Path("/content/lunch-and-learn-data-cleansing")  # Google Colab
    if not colab_copy.exists():
        if "YOUR-ORG" in REPO_URL:
            raise RuntimeError("The instructor needs to set REPO_URL (see README: 'One-time setup').")
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, str(colab_copy)], check=True)
    return colab_copy

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import env
PATHS = env.prepare(SESSION)

In [ ]:
from datetime import time
from decimal import Decimal
import pandas as pd
from lnl import check, conversions, dates, pipeline, trust
from lnl.check import ___

df = pipeline.clean_through(3)
df.dtypes.to_frame("type now")

## 1. Not every number is a number
TDCJ and SID numbers are **identifiers**. You never add or average them, and converting them to integers destroys the leading zeros. **They stay text.** Rule of thumb: if math on it is meaningless, it isn't a number.

## 2. Integers: `disciplinary_points`

In [ ]:
df.loc[df["disciplinary_points"].str.fullmatch(r"0?3(\.0)?( pts)?|three", na=False), "disciplinary_points"].value_counts()

Five ways to write *three*. The conversion strips `pts`, turns number words into digits, and **rejects fractions** instead of rounding them. A score of 2.5 isn't something to fix quietly; it's a question for whoever entered it.

In [ ]:
points, failed = conversions.to_integer(df["disciplinary_points"])
print("Type:", points.dtype, "| failed:", failed.sum())
df.loc[failed, "disciplinary_points"].value_counts()

🧪 **Your turn #1.** What's the average number of disciplinary points, to 2 decimal places?

In [ ]:
answer_1 = ___      # points.mean(), then round(..., 2)
check.s4_ex1(answer_1)

## 3. Money: `restitution_owed`, and why it's never a float

In [ ]:
print(0.1 + 0.2)                            # a float: close, but not exact
print(Decimal("0.10") + Decimal("0.20"))    # a decimal: exact

Floats are approximations. Across thousands of rows the pennies add up to totals that won't reconcile with the finance system. Here are the money formats in this file (the same shape profiling from session 3):

In [ ]:
dates.profile_shapes(df["restitution_owed"])

The rules: remove `$`, `USD`, and commas. `(12.00)` means **negative** (a credit). A lone `-` means **zero**, which is an accounting convention; confirm it with finance before relying on it.

In [ ]:
money, failed = conversions.to_money(df["restitution_owed"])
print("Type:", money.dtype, "| failed:", failed.sum())
df.loc[failed, "restitution_owed"].value_counts()

`$1,2O0.00` has a **letter O** where a zero belongs. A person can guess what was meant; code shouldn't. It goes on the exceptions list.

🧪 **Your turn #2.** What's the total restitution owed?

In [ ]:
answer_2 = ___      # money.sum()
check.s4_ex2(answer_2)

## 4. Time of day: `intake_time`

In [ ]:
dates.profile_shapes(df["intake_time"])

In [ ]:
intake, failed = conversions.to_time(df["intake_time"])
print("Type:", intake.dtype, "| failed:", failed.sum())
display(df.loc[failed, "intake_time"].value_counts())

overnight = (intake >= time(22, 0)) | (intake < time(6, 0))
print("Intakes between 10 PM and 6 AM:", overnight.sum())

`25:10` and `12:75` have the right shape but aren't real times, and `noon` is a word. As text, "between 10 PM and 6 AM" couldn't be answered at all.

## 5. Yes or no: `escape_risk`

In [ ]:
df["escape_risk"].str.upper().value_counts()

🧪 **Your turn #3.** Finish the two lists so every spelling of yes and no gets mapped. (A checkbox-style `X` means yes.)

In [ ]:
TRUE_VALUES  = {"Y", "YES", ___}      # add the other spellings that mean yes
FALSE_VALUES = {"N", "NO", ___}       # add the other spellings that mean no
check.s4_ex3(TRUE_VALUES, FALSE_VALUES)

In [ ]:
escape, unmapped = conversions.to_boolean(df["escape_risk"])
escape.value_counts(dropna=False)

`MAYBE`, `?`, and `PENDING` are genuinely unknown, so they stay null. Never force an unknown into True or False.

## 6. Single-character flags: Y/N and T/F
Most source systems store yes/no as one character. It looks too simple to go wrong. Here are two:

In [ ]:
for col in ["protective_custody", "dampener_required"]:
    display(df[col].value_counts(dropna=False).to_frame(col))

Three ways single-character flags go wrong:

- **Two conventions in one column.** `protective_custody` has Y/N *and* T/F. Records from a legacy system and a newer system were merged into one table, and each system had its own habit. `== "Y"` silently skips every `T`.
- **Lowercase.** `y` is not equal to `Y`.
- **The `astype(bool)` trap.** Python treats any non-empty text as True, including `"N"` and `"F"`:

In [ ]:
print('bool("N") =', bool("N"), '   bool("F") =', bool("F"))
looks_right = df["protective_custody"].dropna().astype(bool)
print(f"astype(bool) puts {looks_right.sum():,} of {len(looks_right):,} records in protective custody")

Every single record, including every `N` and `F`, with no error and no warning. The fix is the same explicit mapping from the escape-risk exercise, which already covers Y/N/T/F in any case:

In [ ]:
flags = {}
for col in ["protective_custody", "dampener_required"]:
    flags[col], leftover = conversions.to_boolean(df[col])
    print(f"{col:20} True={flags[col].sum():,}  False={(~flags[col]).sum():,}  unmapped={leftover.value_counts().to_dict()}")

print()
print(f'Protective custody counting only "Y": {(df["protective_custody"] == "Y").sum():,}')
print(f'Protective custody after mapping:     {flags["protective_custody"].sum():,}')

`?` has no yes-or-no meaning, so it stays null and goes on the exceptions list.

🧪 **Your turn #4.** How many records require a power dampener?

In [ ]:
answer_4 = ___      # flags["dampener_required"] holds True/False; add up the Trues
check.s4_ex4(answer_4)

## 7. Codes with an order: categories
As plain text, custody levels sort alphabetically and anything can be typed in. As an **ordered category**, they sort and compare correctly, and only valid codes are allowed.

In [ ]:
typed = conversions.to_categories(df)
print(typed["custody_level"].cat.categories.tolist())
print("Records at G4 or above:", (typed["custody_level"] >= "G4").sum())

## 8. Everything at once

In [ ]:
df4 = conversions.session4(df)
display(conversions.conversion_report(df))
now = df4.dtypes.astype(str)
now["tdcj_number"] = f"{df4.index.dtype} (the index)"
columns = ["tdcj_number", "date_of_birth", "intake_time", "disciplinary_points", "restitution_owed",
           "escape_risk", "protective_custody", "custody_level", "threat_level"]
pd.DataFrame({"session 1 (as loaded)": pipeline.load_naive().dtypes.astype(str), "now": now}).loc[columns]

### The exceptions list, keyed by TDCJ number
Counts tell you how big a problem is. The data owner needs to know **which records** to fix. Because `tdcj_number` is the ID, every value that didn't convert comes back with the exact record it belongs to:

In [ ]:
issues = conversions.exceptions(df)
print(f"{len(issues):,} values to send back to the data owner")
display(issues["column"].value_counts().to_frame("values"))
issues.head(8)

This is the list you hand back, and it's why the ID matters: "206 points didn't convert" starts a conversation, "TDCJ #01003833 has 2.5 points" gets it fixed. `?` in `dampener_required` and `MAYBE` in `escape_risk` are on it too: unknowns stay null in the data, and the list says where they are.

## 9. The payoff: rerun the Trust Test

In [ ]:
naive = trust.naive_answers(pipeline.load_naive())
now = trust.typed_answers(df4)
trust.scorecard(naive, now)

**Every answer changed.** Q7 and Q8 are the sneaky ones: counting only `"Y"` looks reasonable and misses every `Yes`, `1`, `TRUE`, `X`, and, for the single-character flag, every `T`, `y`, and `t`.

### Questions you couldn't ask on day one

In [ ]:
trusted = df4[df4["dob_flag"].isna()]      # leave impossible birth dates out of averages
display(trusted.groupby("custody_level", observed=True)["age_at_sentencing"].mean().round(1).to_frame("average age at sentencing"))
display(df4.groupby("violence_code", observed=True)["restitution_owed"].sum().to_frame("total restitution"))
display(df4.groupby("threat_level", observed=True)["escape_risk"].mean().round(3).to_frame("share flagged as escape risk"))

## 10. Save with the types intact

In [ ]:
import pyarrow.parquet as pq

path = PATHS["dir"] / "session4_typed.parquet"
df4.to_parquet(path)                             # keeps the TDCJ number as the index
schema = pq.read_schema(path)                    # what's actually stored in the file
pd.Series({col: str(schema.field(col).type) for col in
           ["tdcj_number", "date_of_birth", "intake_time", "restitution_owed", "escape_risk", "custody_level"]},
          name="type stored in the Parquet file").to_frame()

The file itself remembers every type: the TDCJ number as text (and as the index when pandas reads it back), real dates, times, exact decimals, booleans, and the category list. A CSV would store all of it as text and we'd be back to session 1. (When pandas reads it back, money arrives as exact Python `Decimal` values.)

## Wrap-up
- IDs are text, and the TDCJ number labels every record. Money is decimal. Codes are categories. Yes/no is boolean with room for unknown.
- Single-character flags: map them explicitly (Y/N **and** T/F, any case). Never `astype(bool)`.
- **Report what didn't convert** instead of guessing (`2.5` points, `$1,2O0.00`, `25:10`, `MAYBE`), record by record, keyed by TDCJ number.
- The same five questions from session 1 now have trustworthy answers, and new questions became possible.

**Next session:** the same pipeline in PySpark, and making sure it agrees with pandas to the cent.

---
## Answer key
```python
answer_1 = round(points.mean(), 2)
answer_2 = money.sum()
TRUE_VALUES  = {"Y", "YES", "1", "TRUE", "T", "X"}
FALSE_VALUES = {"N", "NO", "0", "FALSE", "F"}
answer_4 = flags["dampener_required"].sum()
```